<div dir="rtl">

# الترم ١ · الوحدة ٣: البيانات والرسوم البيانية

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahman-abozarifa04/ai-for-kids-course/blob/main/notebooks/teens/term-1/02-data-and-charts.ipynb)

البيانات هي **وقود** الذكاء الاصطناعي. النهارده هنتعلم:
1. نحمّل البيانات في **pandas**.
2. ننضّفها (قيم ناقصة، تكرار، أخطاء).
3. نرسمها بـ **Matplotlib** و **Seaborn** علشان نشوف الأنماط.

البيانات: استبيان خيالي لـ ٤٠ طالب ثانوي في مصر عن ساعات النوم والمذاكرة والموبايل والدرجات.

</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

<div dir="rtl">

## ١. نعمل البيانات

في الحقيقة بنقرا البيانات من ملف بـ `pd.read_csv("file.csv")`. هنا هنولّدها بالكود علشان النوتبوك يشتغل من غير ملفات.

</div>

In [ ]:
rng = np.random.default_rng(42)
n = 40
study = rng.uniform(0.5, 6, n).round(1)
sleep = rng.uniform(4, 9, n).round(1)
phone = rng.uniform(1, 8, n).round(1)
score = (40 + 7 * study + 3 * sleep - 2.5 * phone + rng.normal(0, 5, n)).clip(0, 100).round()
city = rng.choice(["Cairo", "Giza", "Alexandria", "Mansoura"], n)

df = pd.DataFrame({"study_hours": study, "sleep_hours": sleep, "phone_hours": phone,
                   "city": city, "score": score})

# نزوّد شوية "وساخة" زي البيانات الحقيقية
df.loc[3, "sleep_hours"] = np.nan          # قيمة ناقصة
df.loc[10, "score"] = 250                  # غلطة في الإدخال
df.loc[[15, 27], "city"] = "cairo"         # كتابة مختلفة
df = pd.concat([df, df.iloc[[5]]], ignore_index=True)  # صف مكرر

df.head()

<div dir="rtl">

## ٢. نتعرف على البيانات

</div>

In [ ]:
print("Shape (rows, columns):", df.shape)
df.info()

In [ ]:
df.describe()

<div dir="rtl">

**لاحظ:** أعلى درجة `250`! والدرجات من ١٠٠. دي غلطة لازم نصلحها.

</div>

<div dir="rtl">

## ٣. تنضيف البيانات (Data Cleaning)

</div>

In [ ]:
print("Missing values:\n", df.isna().sum())
print("Duplicated rows:", df.duplicated().sum())
print("Cities:", df["city"].unique())

In [ ]:
clean = df.copy()
clean = clean.drop_duplicates()
clean["city"] = clean["city"].str.title()                       # cairo -> Cairo
clean = clean[clean["score"] <= 100]                            # نشيل الدرجات المستحيلة
clean["sleep_hours"] = clean["sleep_hours"].fillna(clean["sleep_hours"].median())

print("Before:", df.shape, "After:", clean.shape)
print("Cities now:", sorted(clean["city"].unique()))
assert clean.isna().sum().sum() == 0

<div dir="rtl">

## ٤. نسأل البيانات أسئلة

</div>

In [ ]:
print("Average score by city:")
print(clean.groupby("city")["score"].mean().round(1).sort_values(ascending=False))

In [ ]:
top = clean.sort_values("score", ascending=False).head(5)
top[["study_hours", "sleep_hours", "phone_hours", "score"]]

<div dir="rtl">

## ٥. نرسم: الصورة بألف رقم

### Histogram: الدرجات متوزعة إزاي؟

</div>

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(clean["score"], bins=10, color="#4F46E5")
plt.title("Distribution of scores")
plt.xlabel("Score")
plt.show()

<div dir="rtl">

### Scatter: هل المذاكرة ليها علاقة بالدرجة؟

</div>

In [ ]:
plt.figure(figsize=(7, 4))
sns.regplot(data=clean, x="study_hours", y="score", color="#059669")
plt.title("Study hours vs score")
plt.show()

<div dir="rtl">

### Heatmap: العلاقات كلها مرة واحدة

الرقم قريب من **+1** يعني لما واحد يزيد التاني بيزيد. قريب من **-1** يعني لما واحد يزيد التاني بيقل.

</div>

In [ ]:
corr = clean[["study_hours", "sleep_hours", "phone_hours", "score"]].corr().round(2)
plt.figure(figsize=(6, 4.5))
sns.heatmap(corr, annot=True, cmap="RdBu", vmin=-1, vmax=1)
plt.title("Correlation")
plt.show()

<div dir="rtl">

### 🏋️ تمرين ١
ارسم **bar chart** لمتوسط ساعات الموبايل في كل مدينة.

<details><summary>الحل</summary>

```python
plt.figure(figsize=(7, 4))
sns.barplot(data=clean, x="city", y="phone_hours", errorbar=None, color="#D97706")
plt.title("Average phone hours by city")
plt.show()
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

### 🏋️ تمرين ٢
اعمل عمود جديد `passed` قيمته `True` لو الدرجة ≥ ٦٠، واحسب نسبة النجاح في الفصل.

<details><summary>الحل</summary>

```python
clean["passed"] = clean["score"] >= 60
print(f"Pass rate: {clean['passed'].mean():.0%}")
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## ⚠️ خلي بالك: العلاقة مش معناها السبب

لو لقينا إن الطلاب اللي بيناموا أكتر درجاتهم أعلى، ده **مش** معناه إن النوم لوحده هو السبب. ممكن يكون فيه عامل تالت (مثلاً: تنظيم الوقت). العلماء بيقولوا: **Correlation is not causation**.

## ✅ لخّص
- `pd.DataFrame` / `read_csv`: نحمّل البيانات.
- `info`, `describe`, `isna`, `duplicated`: نفحصها.
- `drop_duplicates`, `fillna`, الفلترة: ننضّفها.
- `groupby`: نسأل أسئلة.
- `histplot`, `regplot`, `heatmap`: نشوف الأنماط.

</div>